# From MovieLens IDs to LightGCN Graph Data

This notebook prepares the inputs required by a graph recommender without training a neural network yet.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from group_movie_recommender.algorithms.graph_data import (
    build_bipartite_graph,
    positive_movie_sets,
    sample_bpr_batch,
)

## 1. Start with positive training edges

These edges represent ratings of at least four. Original MovieLens IDs are identifiers, not embedding row numbers.

In [ ]:
train_edges = pd.DataFrame(
    {
        "userId": [20, 10, 10, 20, 30],
        "movieId": [200, 100, 200, 300, 100],
    }
)
train_edges

## 2. Create contiguous indices

Users become indices `0, 1, 2` and movies independently become local indices `0, 1, 2`. Movie graph nodes are offset by the number of users, so their global node indices are `3, 4, 5`.

In [ ]:
graph = build_bipartite_graph(train_edges)

user_mapping = pd.DataFrame(
    {"userIndex": range(graph.num_users), "userId": graph.user_ids}
)
movie_mapping = pd.DataFrame(
    {
        "movieIndex": range(graph.num_movies),
        "movieNodeIndex": range(graph.num_users, graph.num_nodes),
        "movieId": graph.movie_ids,
    }
)
display(user_mapping)
display(movie_mapping)

## 3. Inspect the undirected edge index

The first half contains user-to-movie edges. The second half reverses every edge so information can propagate in both directions.

In [ ]:
pd.DataFrame(graph.edge_index, index=["sourceNode", "targetNode"] )

## 4. Sample BPR training triples

BPR compares one observed positive movie with one unseen movie for the same user. The desired relationship is `score(user, positive) > score(user, negative)`.

In [ ]:
batch = sample_bpr_batch(graph, batch_size=8, random_seed=7)
decoded_batch = batch.copy()
decoded_batch["userId"] = graph.original_user_ids(batch["userIndex"].to_numpy())
decoded_batch["positiveMovieId"] = graph.original_movie_ids(
    batch["positiveMovieIndex"].to_numpy()
)
decoded_batch["negativeMovieId"] = graph.original_movie_ids(
    batch["negativeMovieIndex"].to_numpy()
)
decoded_batch

## 5. Verify that negatives are unseen

This check is central to implicit-feedback training. It does not claim that an unseen movie is disliked; it only provides a relative training comparison.

In [ ]:
positive_sets = positive_movie_sets(graph)
checks = [
    row.negativeMovieIndex not in positive_sets[row.userIndex]
    for row in batch.itertuples(index=False)
]
all(checks)

## Next step

The next module will initialize user and movie embeddings, propagate them over `edge_index`, and optimize the BPR objective using these sampled triples.